In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

from mplsoccer import Sbopen
import matplotlib.pyplot as plt

# Project paths
PROJECT_ROOT = Path.cwd().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES = PROJECT_ROOT / "outputs" / "figures"

# Allow notebook to import from src/
sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/Users/tstanton/Desktop/soccer-event-data/soccer-xT-project')

In [2]:
from src.loaders.statsbomb_loader import (
    convert_statsbomb_to_standard
)

In [3]:
parser = Sbopen()

matches = parser.match(
    competition_id=43,
    season_id=3
)

matches[
    [
        'match_id',
        'home_team_name',
        'away_team_name'
    ]
].head()

,match_id,home_team_name,away_team_name
0,7585,Colombia,England
1,7570,England,Belgium
2,7586,Sweden,Switzerland
3,7557,Iran,Portugal
4,7542,Portugal,Morocco


In [4]:
match_id = matches.iloc[0]['match_id']

events, related, freeze, tactics = parser.event(match_id)

events.head()

,id,index,period,timestamp,minute,second,possession,duration,match_id,type_id,...,foul_committed_card_id,foul_committed_card_name,foul_committed_penalty,foul_won_penalty,block_offensive,substitution_replacement_id,substitution_replacement_name,shot_first_time,pass_goal_assist,shot_deflected
0,de3be98d-e227-475b-bd55-f57a6a89d308,1,1,00:00:00,0,0,1,0.000,7585,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,f50ccda4-b768-4f07-9136-8f79fd17dac5,2,1,00:00:00,0,0,1,0.754,7585,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,b5e98805-0a22-4a5e-a306-7d40651a0f6e,3,1,00:00:00,0,0,1,9.320,7585,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,762b829f-5f24-4dd7-bfe2-da7e289838bb,4,1,00:00:00,0,0,1,9.053,7585,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,d4883f20-ce68-4f84-b26a-a049a13cb6be,5,1,00:00:00.240000,0,0,2,0.240,7585,30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
standard_events = convert_statsbomb_to_standard(events)

standard_events.head()

,match_id,period,timestamp_seconds,team,player,event_type,start_x,start_y,end_x,end_y,outcome,possession_id,source_event_id,source
0,7585,1,0.00,Colombia,NaN,starting xi,NaN,NaN,NaN,NaN,successful,1,de3be98d-e227-475b-bd55-f57a6a89d308,statsbomb
1,7585,1,0.00,England,NaN,starting xi,NaN,NaN,NaN,NaN,successful,1,f50ccda4-b768-4f07-9136-8f79fd17dac5,statsbomb
2,7585,1,0.00,England,NaN,half start,NaN,NaN,NaN,NaN,successful,1,b5e98805-0a22-4a5e-a306-7d40651a0f6e,statsbomb
3,7585,1,0.00,Colombia,NaN,half start,NaN,NaN,NaN,NaN,successful,1,762b829f-5f24-4dd7-bfe2-da7e289838bb,statsbomb
4,7585,1,0.24,Colombia,Radamel Falcao García Zárate,pass,60.0,40.0,50.0,41.0,successful,2,d4883f20-ce68-4f84-b26a-a049a13cb6be,statsbomb


In [6]:
from src.processing.xt_actions import extract_xt_actions

In [7]:
xt_actions = extract_xt_actions(standard_events)

xt_actions.head()

,match_id,period,timestamp_seconds,team,player,event_type,start_x,start_y,end_x,end_y,outcome,possession_id,source_event_id,source
4,7585,1,0.24,Colombia,Radamel Falcao García Zárate,pass,60.0,40.0,50.0,41.0,successful,2,d4883f20-ce68-4f84-b26a-a049a13cb6be,statsbomb
6,7585,1,0.48,Colombia,Juan Fernando Quintero Paniagua,carry,50.0,41.0,51.0,40.0,successful,2,b948f032-4c54-4782-a71a-ffeed8908d00,statsbomb
8,7585,1,2.12,Colombia,Juan Fernando Quintero Paniagua,pass,51.0,40.0,47.0,54.0,successful,2,9bdb71f9-c87b-4a66-96f0-def5312ca921,statsbomb
10,7585,1,3.44,Colombia,Carlos Alberto Sánchez Moreno,carry,47.0,54.0,49.0,55.0,successful,2,2ffa2904-8b47-4817-af26-aa9ac8d2881a,statsbomb
11,7585,1,4.20,Colombia,Carlos Alberto Sánchez Moreno,pass,49.0,55.0,65.0,79.0,successful,2,6cb0d85d-bd14-42e3-9c2d-7f99ce437796,statsbomb


In [8]:
xt_actions["event_type"].value_counts()

pass     1161
carry     851
Name: event_type, dtype: int64

In [9]:
xt_actions_path = DATA_PROCESSED / f"xt_actions_match_{match_id}.csv"

xt_actions.to_csv(xt_actions_path, index=False)

xt_actions_path

PosixPath('/Users/tstanton/Desktop/soccer-event-data/soccer-xT-project/data/processed/xt_actions_match_7585.csv')

In [10]:
from src.analytics.xt_model import (
    add_simple_xt,
    player_xt_table
)

In [11]:
xt_actions_with_values = add_simple_xt(xt_actions)

xt_actions_with_values.head()

,match_id,period,timestamp_seconds,team,player,event_type,start_x,start_y,end_x,end_y,...,possession_id,source_event_id,source,start_x_bin,start_y_bin,end_x_bin,end_y_bin,start_xT,end_xT,xT_added
4,7585,1,0.24,Colombia,Radamel Falcao García Zárate,pass,60.0,40.0,50.0,41.0,...,2,d4883f20-ce68-4f84-b26a-a049a13cb6be,statsbomb,5,3,4,4,0.164545,0.133636,-0.030909
6,7585,1,0.48,Colombia,Juan Fernando Quintero Paniagua,carry,50.0,41.0,51.0,40.0,...,2,b948f032-4c54-4782-a71a-ffeed8908d00,statsbomb,4,4,4,3,0.133636,0.133636,0.000000
8,7585,1,2.12,Colombia,Juan Fernando Quintero Paniagua,pass,51.0,40.0,47.0,54.0,...,2,9bdb71f9-c87b-4a66-96f0-def5312ca921,statsbomb,4,3,4,5,0.133636,0.133636,0.000000
10,7585,1,3.44,Colombia,Carlos Alberto Sánchez Moreno,carry,47.0,54.0,49.0,55.0,...,2,2ffa2904-8b47-4817-af26-aa9ac8d2881a,statsbomb,4,5,4,5,0.133636,0.133636,0.000000
11,7585,1,4.20,Colombia,Carlos Alberto Sánchez Moreno,pass,49.0,55.0,65.0,79.0,...,2,6cb0d85d-bd14-42e3-9c2d-7f99ce437796,statsbomb,4,5,6,7,0.133636,0.195455,0.061818


In [12]:
player_xt = player_xt_table(xt_actions_with_values)

player_xt.head(20)

,player,team,total_xT,actions,avg_xT_per_action
18,Jordan Pickford,England,4.018182,66,0.060882
7,David Ospina Ramírez,Colombia,3.801818,28,0.135779
8,Davinson Sánchez Mina,Colombia,2.936364,118,0.024884
16,John Stones,England,2.874545,154,0.018666
11,Harry Maguire,England,2.410909,129,0.018689
19,Juan Fernando Quintero Paniagua,Colombia,1.854545,81,0.022896
1,Ashley Young,England,1.823636,70,0.026052
28,Wílmar Enrique Barrios Terán,Colombia,1.452727,80,0.018159
15,Johan Andrés Mojica Palacio,Colombia,1.421818,115,0.012364
29,Yerry Fernando Mina González,Colombia,1.205455,107,0.011266


In [13]:
player_xt_path = OUTPUT_TABLES / f"player_xt_match_{match_id}.csv"

player_xt.to_csv(player_xt_path, index=False)

player_xt_path

PosixPath('/Users/tstanton/Desktop/soccer-event-data/soccer-xT-project/outputs/tables/player_xt_match_7585.csv')

In [14]:
import numpy as np
np.__version__

'1.26.4'

In [15]:
import socceraction.spadl as spadl
import socceraction.xthreat as xthreat

In [16]:
home_team = matches.iloc[0]["home_team_name"]

home_team_id = events.loc[
    events["team_name"] == home_team,
    "team_id"
].dropna().iloc[0]

home_team, home_team_id

('Colombia', 769)

In [17]:
spadl_actions = spadl.statsbomb.convert_to_actions(
    events,
    home_team_id=home_team_id
)

spadl_actions.head()

KeyError: 'extra'

In [18]:
from socceraction.data.statsbomb import StatsBombLoader
import socceraction.spadl as spadl

In [19]:
SBL = StatsBombLoader(getter="remote", creds=None)

competitions_sa = SBL.competitions()
competitions_sa.head()

,season_id,competition_id,competition_name,country_name,competition_gender,season_name
0,281,9,1. Bundesliga,Germany,male,2023/2024
1,27,9,1. Bundesliga,Germany,male,2015/2016
2,107,1267,African Cup of Nations,Africa,male,2023
3,4,16,Champions League,Europe,male,2018/2019
4,1,16,Champions League,Europe,male,2017/2018


In [20]:
competitions_sa[
    ['competition_id', 'season_id', 'competition_name', 'season_name']
].head(30)

,competition_id,season_id,competition_name,season_name
0,9,281,1. Bundesliga,2023/2024
1,9,27,1. Bundesliga,2015/2016
2,1267,107,African Cup of Nations,2023
3,16,4,Champions League,2018/2019
4,16,1,Champions League,2017/2018
5,16,2,Champions League,2016/2017
6,16,27,Champions League,2015/2016
7,16,26,Champions League,2014/2015
8,16,25,Champions League,2013/2014
9,16,24,Champions League,2012/2013


In [21]:
games_sa = SBL.games(competition_id=43, season_id=3)

games_sa.head()

,game_id,season_id,competition_id,competition_stage,game_day,game_date,home_team_id,away_team_id,home_score,away_score,venue,referee
0,7585,3,43,Round of 16,4,2018-07-03 20:00:00,769,768,1,1,Otkritie Bank Arena,Mark Geiger
1,7570,3,43,Group Stage,3,2018-06-28 20:00:00,768,782,0,1,Stadion Kaliningrad,Damir Skomina
2,7586,3,43,Round of 16,4,2018-07-03 16:00:00,790,773,1,0,Saint-Petersburg Stadium,Damir Skomina
3,7557,3,43,Group Stage,3,2018-06-25 20:00:00,797,780,1,1,Mordovia Arena,Enrique Cáceres
4,7542,3,43,Group Stage,2,2018-06-20 14:00:00,780,788,1,0,Stadion Luzhniki,Mark Geiger


In [22]:
game_id = games_sa.iloc[0]["game_id"]

events_sa = SBL.events(game_id)

events_sa.head()

,game_id,event_id,period_id,team_id,player_id,type_id,type_name,index,timestamp,minute,...,team_name,duration,extra,related_events,player_name,position_id,position_name,location,under_pressure,counterpress
0,7585,de3be98d-e227-475b-bd55-f57a6a89d308,1,769,NaN,35,Starting XI,1,1900-01-01 00:00:00.000,0,...,Colombia,0.000,"{'tactics': {'formation': 433, 'lineup': [{'pl...",[],NaN,NaN,NaN,NaN,False,False
1,7585,f50ccda4-b768-4f07-9136-8f79fd17dac5,1,768,NaN,35,Starting XI,2,1900-01-01 00:00:00.000,0,...,England,0.754,"{'tactics': {'formation': 352, 'lineup': [{'pl...",[],NaN,NaN,NaN,NaN,False,False
2,7585,b5e98805-0a22-4a5e-a306-7d40651a0f6e,1,768,NaN,18,Half Start,3,1900-01-01 00:00:00.000,0,...,England,9.320,{},[762b829f-5f24-4dd7-bfe2-da7e289838bb],NaN,NaN,NaN,NaN,False,False
3,7585,762b829f-5f24-4dd7-bfe2-da7e289838bb,1,769,NaN,18,Half Start,4,1900-01-01 00:00:00.000,0,...,Colombia,9.053,{},[b5e98805-0a22-4a5e-a306-7d40651a0f6e],NaN,NaN,NaN,NaN,False,False
4,7585,d4883f20-ce68-4f84-b26a-a049a13cb6be,1,769,3445.0,30,Pass,5,1900-01-01 00:00:00.240,0,...,Colombia,0.240,"{'pass': {'recipient': {'id': 5692, 'name': 'J...",[5fc9acb8-88c3-4cfb-ad9c-fc250c0dffde],Radamel Falcao García Zárate,24.0,Left Center Forward,"[60.0, 40.0]",False,False


In [23]:
home_team_id_sa = games_sa.iloc[0]["home_team_id"]

home_team_id_sa

769

In [24]:
spadl_actions = spadl.statsbomb.convert_to_actions(
    events_sa,
    home_team_id=home_team_id_sa
)

spadl_actions.head()

,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,type_id,result_id,bodypart_id,action_id
0,7585,d4883f20-ce68-4f84-b26a-a049a13cb6be,1,0.0,769,3445.0,52.058824,34.430380,43.235294,33.569620,0,1,4,0
1,7585,b948f032-4c54-4782-a71a-ffeed8908d00,1,0.0,769,5692.0,43.235294,33.569620,44.117647,34.430380,21,1,0,1
2,7585,9bdb71f9-c87b-4a66-96f0-def5312ca921,1,2.0,769,5692.0,44.117647,34.430380,40.588235,22.379747,0,1,4,2
3,7585,2ffa2904-8b47-4817-af26-aa9ac8d2881a,1,3.0,769,5685.0,40.588235,22.379747,42.352941,21.518987,21,1,0,3
4,7585,6cb0d85d-bd14-42e3-9c2d-7f99ce437796,1,4.0,769,5685.0,42.352941,21.518987,56.470588,0.860759,0,1,5,4


In [25]:
spadl_actions_named = spadl.add_names(spadl_actions)

spadl_actions_named.head()

,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,type_id,result_id,bodypart_id,action_id,type_name,result_name,bodypart_name
0,7585,d4883f20-ce68-4f84-b26a-a049a13cb6be,1,0.0,769,3445.0,52.058824,34.430380,43.235294,33.569620,0,1,4,0,pass,success,foot_left
1,7585,b948f032-4c54-4782-a71a-ffeed8908d00,1,0.0,769,5692.0,43.235294,33.569620,44.117647,34.430380,21,1,0,1,dribble,success,foot
2,7585,9bdb71f9-c87b-4a66-96f0-def5312ca921,1,2.0,769,5692.0,44.117647,34.430380,40.588235,22.379747,0,1,4,2,pass,success,foot_left
3,7585,2ffa2904-8b47-4817-af26-aa9ac8d2881a,1,3.0,769,5685.0,40.588235,22.379747,42.352941,21.518987,21,1,0,3,dribble,success,foot
4,7585,6cb0d85d-bd14-42e3-9c2d-7f99ce437796,1,4.0,769,5685.0,42.352941,21.518987,56.470588,0.860759,0,1,5,4,pass,success,foot_right


In [26]:
spadl_actions_named["type_name"].value_counts()

pass                1042
dribble              908
clearance             50
throw_in              43
bad_touch             36
foul                  36
take_on               29
shot                  28
tackle                27
freekick_crossed      22
cross                 18
goalkick              12
interception          12
shot_penalty          11
freekick_short         8
corner_crossed         7
keeper_save            4
keeper_claim           2
shot_freekick          2
keeper_punch           1
Name: type_name, dtype: int64

In [27]:
xt = xthreat.ExpectedThreat(
    l=16,
    w=12
)

In [41]:
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

xt_model_path = MODEL_DIR / "socceraction_xt_model.json"

xt.save_model(xt_model_path)

xt_model_path

PosixPath('/Users/tstanton/Desktop/soccer-event-data/soccer-xT-project/models/socceraction_xt_model.json')

In [28]:
xt.fit(spadl_actions_named)

# iterations:  34


In [42]:
static_xt_model = xthreat.load_model(xt_model_path)

In [43]:
valued_actions_public = spadl_actions_named.copy()

valued_actions_public["xT_added"] = static_xt_model.rate(
    valued_actions_public
)

valued_actions_public[
    [
        "player_id",
        "type_name",
        "start_x",
        "start_y",
        "end_x",
        "end_y",
        "xT_added"
    ]
].head()

,player_id,type_name,start_x,start_y,end_x,end_y,xT_added
0,3445.0,pass,52.058824,34.430380,43.235294,33.569620,-0.000475
1,5692.0,dribble,43.235294,33.569620,44.117647,34.430380,0.000019
2,5692.0,pass,44.117647,34.430380,40.588235,22.379747,-0.000313
3,5685.0,dribble,40.588235,22.379747,42.352941,21.518987,0.000000
4,5685.0,pass,42.352941,21.518987,56.470588,0.860759,0.000125


In [29]:
spadl_actions_named["xT_value"] = xt.rate(
    spadl_actions_named
)

spadl_actions_named.head()

,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,type_id,result_id,bodypart_id,action_id,type_name,result_name,bodypart_name,xT_value
0,7585,d4883f20-ce68-4f84-b26a-a049a13cb6be,1,0.0,769,3445.0,52.058824,34.430380,43.235294,33.569620,0,1,4,0,pass,success,foot_left,-0.000475
1,7585,b948f032-4c54-4782-a71a-ffeed8908d00,1,0.0,769,5692.0,43.235294,33.569620,44.117647,34.430380,21,1,0,1,dribble,success,foot,0.000019
2,7585,9bdb71f9-c87b-4a66-96f0-def5312ca921,1,2.0,769,5692.0,44.117647,34.430380,40.588235,22.379747,0,1,4,2,pass,success,foot_left,-0.000313
3,7585,2ffa2904-8b47-4817-af26-aa9ac8d2881a,1,3.0,769,5685.0,40.588235,22.379747,42.352941,21.518987,21,1,0,3,dribble,success,foot,0.000000
4,7585,6cb0d85d-bd14-42e3-9c2d-7f99ce437796,1,4.0,769,5685.0,42.352941,21.518987,56.470588,0.860759,0,1,5,4,pass,success,foot_right,0.000125


In [30]:
spadl_actions_named.sort_values(
    "xT_value",
    ascending=False
)[
    [
        "player_id",
        "type_name",
        "start_x",
        "start_y",
        "end_x",
        "end_y",
        "xT_value"
    ]
].head(20)

,player_id,type_name,start_x,start_y,end_x,end_y,xT_value
1708,3205.0,pass,77.647059,56.810127,102.352941,30.126582,0.227550
846,3205.0,pass,79.411765,51.645570,100.588235,32.708861,0.212217
1583,5691.0,cross,103.235294,61.974684,98.823529,30.987342,0.125433
2283,5691.0,dribble,90.882353,55.088608,102.352941,56.810127,0.103765
1581,5691.0,dribble,97.058824,43.037975,99.705882,66.278481,0.089076
379,3468.0,pass,91.764706,23.240506,81.176471,51.645570,0.031729
85,3532.0,dribble,78.529412,50.784810,79.411765,51.645570,0.031529
920,3468.0,pass,94.411765,36.151899,82.941176,55.949367,0.030529
803,3308.0,pass,71.470588,56.810127,81.176471,53.367089,0.030413
105,3205.0,dribble,81.176471,60.253165,83.823529,55.949367,0.026267


In [31]:
player_lookup = events_sa[
    ["player_id", "player_name"]
].drop_duplicates()

player_lookup.head()

,player_id,player_name
0,NaN,NaN
4,3445.0,Radamel Falcao García Zárate
5,5692.0,Juan Fernando Quintero Paniagua
7,10955.0,Harry Kane
9,5685.0,Carlos Alberto Sánchez Moreno


In [32]:
spadl_actions_named = spadl_actions_named.merge(
    player_lookup,
    on="player_id",
    how="left"
)

spadl_actions_named.head()

,game_id,original_event_id,period_id,time_seconds,team_id,player_id,start_x,start_y,end_x,end_y,type_id,result_id,bodypart_id,action_id,type_name,result_name,bodypart_name,xT_value,player_name
0,7585,d4883f20-ce68-4f84-b26a-a049a13cb6be,1,0.0,769,3445.0,52.058824,34.430380,43.235294,33.569620,0,1,4,0,pass,success,foot_left,-0.000475,Radamel Falcao García Zárate
1,7585,b948f032-4c54-4782-a71a-ffeed8908d00,1,0.0,769,5692.0,43.235294,33.569620,44.117647,34.430380,21,1,0,1,dribble,success,foot,0.000019,Juan Fernando Quintero Paniagua
2,7585,9bdb71f9-c87b-4a66-96f0-def5312ca921,1,2.0,769,5692.0,44.117647,34.430380,40.588235,22.379747,0,1,4,2,pass,success,foot_left,-0.000313,Juan Fernando Quintero Paniagua
3,7585,2ffa2904-8b47-4817-af26-aa9ac8d2881a,1,3.0,769,5685.0,40.588235,22.379747,42.352941,21.518987,21,1,0,3,dribble,success,foot,0.000000,Carlos Alberto Sánchez Moreno
4,7585,6cb0d85d-bd14-42e3-9c2d-7f99ce437796,1,4.0,769,5685.0,42.352941,21.518987,56.470588,0.860759,0,1,5,4,pass,success,foot_right,0.000125,Carlos Alberto Sánchez Moreno


In [33]:
player_xt = (
    spadl_actions_named
    .groupby("player_name", as_index=False)
    .agg(
        total_xT=("xT_value", "sum"),
        actions=("xT_value", "count"),
        avg_xT=("xT_value", "mean")
    )
    .sort_values("total_xT", ascending=False)
)

player_xt.head(20)

,player_name,total_xT,actions,avg_xT
22,Kyle Walker,0.286934,105,0.002733
15,Johan Andrés Mojica Palacio,0.161086,89,0.001810
8,Davinson Sánchez Mina,0.030500,105,0.000290
0,Andrés Mateus Uribe Villa,0.027575,35,0.000788
23,Luis Fernando Muriel Fruto,0.027034,22,0.001229
25,Radamel Falcao García Zárate,0.022840,57,0.000401
19,Juan Fernando Quintero Paniagua,0.015610,69,0.000226
29,Yerry Fernando Mina González,0.014447,100,0.000144
28,Wílmar Enrique Barrios Terán,0.012986,74,0.000175
27,Santiago Arias Naranjo,0.006931,76,0.000091


In [34]:
from src.analytics.static_xt import add_static_xt, player_xt_table

In [35]:
static_xt_grid = np.array([
    [0.001, 0.002, 0.003, 0.004, 0.006, 0.008, 0.011, 0.015, 0.020, 0.027, 0.035, 0.045, 0.060, 0.080, 0.110, 0.140],
    [0.001, 0.002, 0.003, 0.005, 0.007, 0.010, 0.014, 0.019, 0.026, 0.035, 0.048, 0.065, 0.090, 0.120, 0.160, 0.210],
    [0.001, 0.002, 0.004, 0.006, 0.009, 0.013, 0.018, 0.025, 0.034, 0.047, 0.065, 0.090, 0.125, 0.170, 0.230, 0.300],
    [0.001, 0.002, 0.004, 0.007, 0.010, 0.015, 0.021, 0.030, 0.042, 0.060, 0.085, 0.120, 0.170, 0.240, 0.330, 0.420],
    [0.001, 0.002, 0.004, 0.007, 0.010, 0.015, 0.021, 0.030, 0.042, 0.060, 0.085, 0.120, 0.170, 0.240, 0.330, 0.420],
    [0.001, 0.002, 0.004, 0.006, 0.009, 0.013, 0.018, 0.025, 0.034, 0.047, 0.065, 0.090, 0.125, 0.170, 0.230, 0.300],
    [0.001, 0.002, 0.003, 0.005, 0.007, 0.010, 0.014, 0.019, 0.026, 0.035, 0.048, 0.065, 0.090, 0.120, 0.160, 0.210],
    [0.001, 0.002, 0.003, 0.004, 0.006, 0.008, 0.011, 0.015, 0.020, 0.027, 0.035, 0.045, 0.060, 0.080, 0.110, 0.140],
])

In [36]:
valued_actions = add_static_xt(
    spadl_actions_named,
    static_xt_grid
)

valued_actions[
    [
        "player_id",
        "type_name",
        "start_x",
        "start_y",
        "end_x",
        "end_y",
        "start_xT",
        "end_xT",
        "xT_added"
    ]
].head()

,player_id,type_name,start_x,start_y,end_x,end_y,start_xT,end_xT,xT_added
0,3445.0,pass,52.058824,34.430380,43.235294,33.569620,0.030,0.021,-0.009
1,5692.0,dribble,43.235294,33.569620,44.117647,34.430380,0.021,0.021,0.000
2,5692.0,pass,44.117647,34.430380,40.588235,22.379747,0.021,0.018,-0.003
3,5685.0,dribble,40.588235,22.379747,42.352941,21.518987,0.018,0.018,0.000
4,5685.0,pass,42.352941,21.518987,56.470588,0.860759,0.018,0.020,0.002


In [38]:
valued_actions.columns.tolist()

['game_id',
 'original_event_id',
 'period_id',
 'time_seconds',
 'team_id',
 'player_id',
 'start_x',
 'start_y',
 'end_x',
 'end_y',
 'type_id',
 'result_id',
 'bodypart_id',
 'action_id',
 'type_name',
 'result_name',
 'bodypart_name',
 'xT_value',
 'player_name_x',
 'start_x_bin',
 'start_y_bin',
 'end_x_bin',
 'end_y_bin',
 'start_xT',
 'end_xT',
 'xT_added',
 'player_name_y']

In [39]:
# Build a clean player lookup
player_lookup = events_sa[
    ["player_id", "player_name"]
].dropna().drop_duplicates()

# Remove any old player_name columns if they exist
valued_actions_clean = valued_actions.drop(
    columns=[col for col in valued_actions.columns if col.startswith("player_name")],
    errors="ignore"
)

# Merge clean names back in
valued_actions_clean = valued_actions_clean.merge(
    player_lookup,
    on="player_id",
    how="left"
)

valued_actions_clean[["player_id", "player_name"]].head()

,player_id,player_name
0,3445.0,Radamel Falcao García Zárate
1,5692.0,Juan Fernando Quintero Paniagua
2,5692.0,Juan Fernando Quintero Paniagua
3,5685.0,Carlos Alberto Sánchez Moreno
4,5685.0,Carlos Alberto Sánchez Moreno


In [40]:
player_static_xt = player_xt_table(
    valued_actions_clean,
    player_col="player_name"
)

player_static_xt.head(20)

,player_name,total_xT,actions,avg_xT
20,Juan Guillermo Cuadrado Bello,3.180,133,0.023910
15,Johan Andrés Mojica Palacio,3.087,130,0.023746
19,Juan Fernando Quintero Paniagua,2.724,89,0.030607
8,Davinson Sánchez Mina,1.135,136,0.008346
0,Andrés Mateus Uribe Villa,0.926,46,0.020130
21,Kieran Trippier,0.799,98,0.008153
7,David Ospina Ramírez,0.787,30,0.026233
4,Carlos Arturo Bacca Ahumada,0.665,34,0.019559
28,Wílmar Enrique Barrios Terán,0.632,96,0.006583
13,Jefferson Andrés Lerma Solís,0.614,54,0.011370
